# LLMRayActor（旨在利用 Ray 框架对 vLLM进行分布式推理封装）
- LLMRayActor：worker端Ray-Actor，它主要是把vllm实例进行了一些包装，包装的目的是为了让ds_rank0和all vllm ranks间可以进行PPO-Actor的权重通讯

In [ ]:
@ray.remote
class LLMRayActor:
    def __init__(self, *args, **kwargs):
        import vllm

        self.__version__ = vllm.__version__
        assert self.__version__ >= "0.4.2", "OpenRLHF only supports vLLM >= 0.4.2"

        noset_visible_devices = kwargs.pop("noset_visible_devices", False)
        self.use_gpu_executor = kwargs["tensor_parallel_size"] == 1 and not noset_visible_devices

        # See https://github.com/vllm-project/vllm/blob/main/vllm/executor/gpu_executor.py
        if self.use_gpu_executor:
            from openrlhf.trainer.ray.vllm_worker_wrap import WorkerWrap

            vllm.worker.worker.Worker = WorkerWrap
        else:
            # RayGPUExecutor
            # See the patch https://github.com/vllm-project/vllm/commit/479d69fad0538f04cb22bf13e76ff91cfeb8a4e5
            kwargs["worker_use_ray"] = True

            if vllm.__version__ > "0.6.4.post1":
                # https://github.com/vllm-project/vllm/pull/10555
                kwargs["worker_cls"] = "openrlhf.trainer.ray.vllm_worker_wrap.WorkerWrap"
            else:
                '''
                vllm.executor.ray_utils -> RayWorkerWrapper
                '''
                RayWorkerWrapperPath = vllm.executor.ray_utils

                class RayWorkerWrapper(RayWorkerWrapperPath.RayWorkerWrapper):
                    def __init__(self, *args, **kwargs) -> None:
                        kwargs["worker_module_name"] = "openrlhf.trainer.ray.vllm_worker_wrap"
                        kwargs["worker_class_name"] = "WorkerWrap"
                        super().__init__(*args, **kwargs)

                RayWorkerWrapperPath.RayWorkerWrapper = RayWorkerWrapper

        self.llm = vllm.LLM(*args, **kwargs)

    
    '''
    init_process_group 方法，它是连接 Ray 分布式框架与 PyTorch 分布式通信 (NCCL) 的关键桥梁。
    
    '''
    def init_process_group(self, master_address, master_port, rank_offset, world_size, group_name, backend):
        '''
        
        根据是单机模式还是 Ray 分布式模式，
        分别调用驱动程序（Driver）或远程工作节点（Workers）的初始化方法，用于多 GPU 或多节点间的通信。
        
        vllm.llm_engine.model_executor
        
        '''
        if self.use_gpu_executor:
            
            return self.llm.llm_engine.model_executor.driver_worker.init_process_group(
                master_address, master_port, rank_offset, world_size, group_name, backend
            )
        else:
            return self.llm.llm_engine.model_executor._run_workers(
                "init_process_group", master_address, master_port, rank_offset, world_size, group_name, backend
            )

    def generate(self, *args, **kwargs):
        return self.llm.generate(*args, **kwargs)
    
    def update_weight(self, name, dtype, shape, empty_cache=False):
        self.stop_remote_worker_execution_loop()

        if self.use_gpu_executor:
            
            return self.llm.llm_engine.model_executor.driver_worker.update_weight(name, dtype, shape, empty_cache)
        else:
            return self.llm.llm_engine.model_executor._run_workers("update_weight", name, dtype, shape, empty_cache)

    def stop_remote_worker_execution_loop(self):
        # Fix error for using 2 communication group
        # https://github.com/vllm-project/vllm/commit/eb6d3c264d0cd8e44dec16bca7947fbe96415ce9#diff-e1ad69e38e033accddfa5480ec808c4740eb39244d1ef51cc3407e20dde8cfd4
        if self.__version__ > "0.4.2":
            self.llm.llm_engine.model_executor.stop_remote_worker_execution_loop()


这段代码定义了一个名为 LLMRayActor 的类，旨在利用 Ray 框架对 vLLM（Vectorized Large Language Model）进行分布式推理封装。该代码通过版本控制和条件逻辑，适配了 vLLM 不同版本之间的 API 变化，特别是针对 Worker 类的替换和执行器（Executor）的选择。

以下是对该代码的详细解析：

## 类定义与装饰器
-@ray.remote: 这是一个 Ray 框架的装饰器，<font color='red'>表明 LLMRayActor 类将作为一个 Ray Actor（参与者）运行。这意味着该类的实例将在 Ray 集群中的某个节点上作为一个独立的进程运行，拥有自己的内存空间，并可以通过 Ray 的 API 进行远程调用。</font>

In [ ]:
@ray.remote
class LLMRayActor:

## 初始化方法 __init__

该方法负责初始化 vLLM 引擎，并根据硬件配置和 vLLM 版本动态调整执行策略。

###  版本检查与配置
- <font color='red'>版本检查：强制要求 vLLM 版本不低于 0.4.2。</font>
- <font color='red'>配置判断：根据 tensor_parallel_size（张量并行大小）和 noset_visible_devices 标志，决定是否使用单 GPU 执行器（use_gpu_executor）。</font>

In [ ]:
def __init__(self, *args, **kwargs):
    import vllm
    self.__version__ = vllm.__version__
    assert self.__version__ >= "0.4.2", "OpenRLHF only supports vLLM >= 0.4.2"
    
    noset_visible_devices = kwargs.pop("noset_visible_devices", False)
    self.use_gpu_executor = kwargs["tensor_parallel_size"] == 1 and not noset_visible_devices

### 执行器与 Worker 适配逻辑

这是代码中最复杂的部分，<font color='red'>用于适配 vLLM 的内部架构变化。</font>

#### <font color='green'>情况 A：使用单 GPU 执行器</font>
- 当仅使用单个 GPU 时，直接替换默认的 Worker 类为自定义的 WorkerWrap（通常用于集成特定的训练或推理逻辑，如 OpenRLHF）。

In [ ]:
if self.use_gpu_executor:
    '''
    WorkerWrap
    '''
    from openrlhf.trainer.ray.vllm_worker_wrap import WorkerWrap
    vllm.worker.worker.Worker = WorkerWrap

#### <font color='green'>情况 B：使用 Ray 分布式执行器 (RayGPUExecutor)</font>
- RayWorkerWrapper 重写 (vLLM <= 0.6.4)：
    - 为了向后兼容，代码动态重写了 RayWorkerWrapper 类。
    - 通过修改 worker_module_name 和 worker_class_name 参数，强制 Ray 在远程节点上实例化自定义的 WorkerWrap 类，而不是默认的 Worker。
- 新式 Worker Class 指定 (vLLM > 0.6.4.post1)：
    - <font color='red'>利用了 vLLM 0.6.5+ 版本引入的新特性（参考 PR #10555）。</font>
    - 直接通过 kwargs["worker_cls"] 参数传入自定义 Worker 的路径，这是一种更简洁、官方支持的扩展方式。

In [ ]:
            # RayGPUExecutor
            # See the patch https://github.com/vllm-project/vllm/commit/479d69fad0538f04cb22bf13e76ff91cfeb8a4e5
            kwargs["worker_use_ray"] = True

            if vllm.__version__ > "0.6.4.post1":
                # https://github.com/vllm-project/vllm/pull/10555
                kwargs["worker_cls"] = "openrlhf.trainer.ray.vllm_worker_wrap.WorkerWrap"
            else:
                '''
                为了向后兼容，代码动态重写了 RayWorkerWrapper 类。
                '''
                RayWorkerWrapperPath = vllm.executor.ray_utils

                class RayWorkerWrapper(RayWorkerWrapperPath.RayWorkerWrapper):
                    def __init__(self, *args, **kwargs) -> None:
                        kwargs["worker_module_name"] = "openrlhf.trainer.ray.vllm_worker_wrap"
                        kwargs["worker_class_name"] = "WorkerWrap"
                        super().__init__(*args, **kwargs)

                RayWorkerWrapperPath.RayWorkerWrapper = RayWorkerWrapper

### LLM 实例化
- 最终根据上述配置创建 vLLM 的 LLM 引擎实例。

In [ ]:
self.llm = vllm.LLM(*args, **kwargs)

## 核心功能方法
###  generate
- 功能：标准的文本生成接口，将参数透传给底层的 vLLM 引擎。

In [ ]:
def generate(self, *args, **kwargs):
    return self.llm.generate(*args, **kwargs)

### init_process_group
- 功能：初始化 PyTorch 分布式进程组。
- 逻辑：<font color='red'>根据是单机模式还是 Ray 分布式模式，分别调用驱动程序（Driver）或远程工作节点（Workers）的初始化方法，用于多 GPU 或多节点间的通信。</font>

In [ ]:
def init_process_group(self, master_address, master_port, rank_offset, world_size, group_name, backend):
    # 根据 use_gpu_executor 分别调用 driver_worker 或 _run_workers

### update_weight 与 stop_remote_worker_execution_loop
- 背景：参考了 vLLM 提交记录 eb6d3c2。在 v0.4.2 之后，为了避免死锁和资源占用，引入了显式停止远程 Worker 执行循环的机制。
- 功能：
    - stop_remote_worker_execution_loop：优雅地停止远程 Worker 上的模型执行循环，释放计算资源，以便进行权重更新（update_weight）或其他控制平面操作（如 LoRA 切换）。

In [ ]:
def update_weight(self, name, dtype, shape, empty_cache=False):
    self.stop_remote_worker_execution_loop()
    # ... 调用底层 update_weight 逻辑

def stop_remote_worker_execution_loop(self):
    if self.__version__ > "0.4.2":
        self.llm.llm_engine.model_executor.stop_remote_worker_execution_loop()

## 总结

这段代码是一个兼容性适配层，主要服务于 OpenRLHF 等需要与 vLLM 深度集成的框架。它解决了以下关键问题：

1. 版本兼容性：无缝支持 vLLM 0.4.2 到 0.6.5+ 的多个版本。
2. 自定义 Worker 注入：无论 vLLM 内部架构如何变化（从修改 RayWrapper 到直接指定 WorkerCls），都能成功注入自定义的 WorkerWrap 逻辑。
3. 资源管理：利用 vLLM 最新的 API 正确管理远程 Worker 的生命周期，确保在权重更新等操作时的稳定性。

# init_process_group()
- <font color='red'>这段代码的核心在于 init_process_group 方法，它是连接 Ray 分布式框架与 PyTorch 分布式通信 (NCCL) 的关键桥梁。</font>

在 OpenRLHF 这种混合训练框架中，<font color='green'>我们需要将负责训练的 Deepspeed Actor 和负责推理的 vLLM Engine 纳入同一个通信组，以便在训练结束后将权重同步给推理引擎。</font>

###### 以下是对这段代码的详细架构级解释：

## 核心背景：为什么需要这个方法？

在 RLHF（人类反馈强化学习）流程中，通常有两个主要角色：
- <font color='red'>训练</font>：Trainer (Actor)：使用 PyTorch/Deepspeed 进行梯度更新。
- <font color='red'>推理</font>：Rollout (vLLM)：使用 vLLM 进行高效的文本生成。

<font color='red'>为了让 vLLM 能够实时获取 Trainer 更新后的权重，它们必须建立通信连接（通常基于 NCCL）。
- <font color='green'>init_process_group 的作用就是让 vLLM 内部的 Worker 进程加入到一个已经存在的 PyTorch 分布式通信组中。</font>

## 代码逻辑深度解析
- 该方法根据 self.use_gpu_executor 的状态，<font color='red'>分两条路径执行，这对应了 vLLM 底层的两种执行模式。


### 路径 A：单 GPU / 本地模式 (self.use_gpu_executor == <font color='red'>True</font>)

In [ ]:
if self.use_gpu_executor:
    return self.llm.llm_engine.model_executor.driver_worker.init_process_group(...)

- 场景：当 <font color='red'>tensor_parallel_size=1 且未强制使用 Ray 时。</font>
- 架构：vLLM 运行在同一个进程中，没有分离的 Ray Worker 进程。
- 执行流：
    - self.llm 是 vLLM 的主入口。
    - llm_engine 是核心调度引擎。
    - model_executor 是执行器。
    - driver_worker 是实际干活的 Worker（在单卡模式下，它就在主进程中）。
- 操作：<font color='red'>直接调用本地对象的 init_process_group 方法。这会在当前进程初始化一个 torch.distributed 的进程组。</font>

### 路径 B：多 GPU / Ray 分布式模式 (self.use_gpu_executor == <font color='red'>False</font>)

In [ ]:
else:
    return self.llm.llm_engine.model_executor._run_workers(
        "init_process_group", master_address, master_port, rank_offset, world_size, group_name, backend
    )

- 场景：当 <font color='red'>tensor_parallel_size > 1 或者显式启用了 Ray 后端时。</font>
- 架构：<font color='red'>vLLM 的 Worker 运行在独立的 Ray Actor 进程中（即 LLMRayActor 内部启动的子 Worker）。</font>
- 关键方法 <font color='red'>_run_workers</font>：
    - 这是一个“广播”或“远程过程调用 (RPC)”机制。
    - <font color='red'>它遍历 model_executor 管理的所有 Worker（例如 TP=4 时会有 4 个 Worker）。</font>
    - <font color='green'>它通过 Ray 的句柄，异步地在每个远程 Worker 进程上执行 init_process_group 函数。</font>
- 参数透传：
    - master_address, master_port: 告诉 vLLM 的 Worker 去哪里找“老大”（通常是 Deepspeed 的 Rank 0 进程）。
    - rank_offset: <font color='red'>非常关键。因为 vLLM 的 Worker 也有自己的 Rank (0, 1, 2...)，为了不和 Trainer 的 Rank 冲突，需要给它们分配一个偏移量（例如 Trainer 用了 0-7，vLLM 就从 8 开始）。</font>
    - world_size: 整个通信组的总进程数。

## 图解通信组构建流程

假设我们有一个 Deepspeed Actor (占用 GPU 0, Rank 0) 和一个 vLLM Engine (占用 GPU 1, TP=2)。

1. Deepspeed 初始化：Rank 0 启动，作为通信组的主节点。
2. 调用 init_process_group：
- Deepspeed 代码调用 LLMRayActor.init_process_group(...)。
3. 进入 else 分支 (Ray 模式)：
- driver_worker (Ray Actor 的管理者) 接收到请求。
- 它调用 _run_workers。
4. 远程执行：
- vLLM <font color='red'>Worker 0 (GPU 1)</font>: 收到指令，连接到 Master，注册自己的 Global Rank (例如 Rank 1)。
- vLLM <font color='red'>Worker 1 (GPU 2)</font>: 收到指令，连接到 Master，注册自己的 Global Rank (例如 Rank 2)。
5. 结果：
- <font color='red'>现在 Rank 0 (Deepspeed), Rank 1 (vLLM-TP0), Rank 2 (vLLM-TP1) 都在同一个 torch.distributed 组里了。</font>
- <font color='green'>接下来就可以使用 broadcast 操作，将 Rank 0 的权重直接发送给 Rank 1 和 Rank 2。</font>

## 总结
这段代码展示了 OpenRLHF 如何“黑盒”操作 vLLM：

1. 统一接口：<font color='red'>无论底层是单卡还是多卡分布式，对外都暴露统一的 init_process_group 接口。</font>
2. 动态路由：利用 use_gpu_executor 标志位，精准地将初始化请求路由到正确的执行单元（本地 Driver 还是远程 Ray Workers）。
3. 分布式融合：通过 rank_offset 和 master_address 等参数，成功将推理引擎“插入”到训练引擎的通信网络中，实现了高效的权重同步。

# vllm.llm_engine.model_executor
- <font color='red'>详见vllm/vllm.llm_engine.model_executor文件夹</font>